In [47]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (Convolutional MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\Convolutional-CIFAR-10"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
BATCH_SIZES = [64, 1024]
PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# =========================
# CREATE OUTPUT DIRECTORIES
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)
print(f"[INFO] Graphs: {AUC_GRAPH_DIR}")
print(f"[INFO] AUC data: {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# Maximum CE cap
CE_MAX = np.log(10)  # ≈2.3026

# Global x cutoff for BOTH plotting and AUC
X_CUTOFF = 400

# =========================
# DEFINE COLORS FOR PERCENTAGES (matches SLP/FMNIST)
# =========================
PRUNING_COLOR_MAP = {
    0.0:  "#17becf",  # 0%
    0.1:  "#B9D9EB",  # 10%
    0.2:  "#bcbd22",  # 20%
    0.3:  "#7f7f7f",  # 30%
    0.4:  "#e377c2",  # 40%
    0.5:  "#8c564b",  # 50%
    0.6:  "#800080",  # 60%
    0.7:  "#d62728",  # 70%
    0.8:  "#2ca02c",  # 80%
    0.9:   "#C5B0D5",
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]
COLOR_LIST = COLOR_LIST[::-1]
# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    # Create subfolders for this prune layer
    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    # Dictionary to store all AUC values for this layer
    auc_table = []

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # -------------------------
        # LOAD AVERAGED FILES
        # -------------------------
        for p in PRUNING_PERCENTAGES:

            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            file_path = os.path.join(
                folder,
                f"averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] File not found: {file_path}")
                continue

            df = pd.read_csv(file_path)

            # Cap CE values
            df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
            df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

            avg_dfs[p] = df

        if not avg_dfs:
            print(f"No data found for {prune_layer}, batch size {bs}")
            continue

        # -------------------------
        # PLOT FUNCTION
        # -------------------------
        def plot_ce(avg_dfs, bs, ce_column, title):

            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | CIFAR-10 | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(title)

            # -----------------------------
            # ln(10) marker (22% from left)
            # -----------------------------
            if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )
            
            for i, p in enumerate(sorted(avg_dfs.keys(), reverse=True)):

                df = avg_dfs[p]
                color = COLOR_LIST[i % len(COLOR_LIST)]

                if p == 1.0:
                    ce_value = df[ce_column].iloc[-1]
                    ce_value = min(ce_value, CE_MAX)

                    # 100% pruning line from x=0 to x=400
                    x_clean = np.array([0, X_CUTOFF])
                    y_clean = np.array([ce_value, ce_value])
                    auc = ce_value * X_CUTOFF

                    plt.plot(
                        x_clean, y_clean,
                        color=color,
                        linewidth=2.5,
                        label=f"P%={int(p*100)} | AUC={auc:.2f}"
                    )

                else:
                    # Truncate strictly at x = 400 for BOTH plotting and AUC
                    df_trunc = df[df["Batch_Number"] <= X_CUTOFF]

                    x = df_trunc["Batch_Number"].to_numpy()
                    y = df_trunc[ce_column].to_numpy()
                    y = np.minimum(y, CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x_clean = x[mask]
                    y_clean = y[mask]

                    if len(x_clean) == 0:
                        print(f"[WARNING] No valid data up to Batch_Number={X_CUTOFF} for P%={p}")
                        continue

                    auc = np.trapezoid(y_clean, x_clean)

                    plt.plot(
                        x_clean, y_clean,
                        color=color,
                        linewidth=2.0,
                        label=f"P%={int(p*100)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(
                        x_clean, y_clean,
                        alpha=0.2,
                        color=color
                    )

                # Save AUC for table
                auc_table.append({
                    "Batch_Size": bs,
                    "Pruning_Percentage": p,
                    "CE_Type": ce_column,
                    "AUC": auc
                })

            plt.xlim(0, X_CUTOFF)
            plt.grid(True)
            plt.tight_layout()
            plt.legend(
                loc='center left',
                bbox_to_anchor=(1, 0.5),
                frameon=False
            )

            # -------------------------
            # SAVE FIGURE
            # -------------------------
            save_name = f"{prune_layer}_{title.replace(' ', '_')}_BS_{bs}.png"
            save_path = os.path.join(layer_graph_dir, save_name)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"[SAVED] {save_path}")

        # -------------------------
        # Plot CE_Train
        # -------------------------
        plot_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE_Train")

        # -------------------------
        # Plot CE_Test
        # -------------------------
        plot_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE_Test")

    # -------------------------
    # SAVE AUC TABLE AS CSV
    # -------------------------
    if auc_table:
        auc_df = pd.DataFrame(auc_table)
        csv_path = os.path.join(layer_data_dir, f"AUC_table_{prune_layer}.csv")
        auc_df.to_csv(csv_path, index=False)
        print(f"[SAVED] AUC table: {csv_path}")

print("\nAll AUC graphs and tables generated successfully.")

[INFO] Graphs: C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100
[INFO] AUC data: C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_data_0-100

Processing prune layer: CONV
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\CONV\CONV_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\CONV\CONV_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\CONV\CONV_Average_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\CONV\CONV_Average_CE_Test_BS_1024.png
[SAVED] AUC table: C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_data_0-100\CONV\AUC_table_CONV.csv

Processing prune layer: FHL
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\FHL\FHL_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-CIFAR-10\AUC_graph_0-100\FHL\FHL_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutiona

In [45]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (Convolutional MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_SIZES = [64, 1024]

# ONLY these pruning percentages will be loaded/plotted
PRUNING_PERCENTAGES = [0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 1.0]

# =========================
# CREATE OUTPUT DIRECTORIES
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_90-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_90-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)
print(f"[INFO] Graphs: {AUC_GRAPH_DIR}")
print(f"[INFO] AUC data: {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# Maximum CE cap
CE_MAX = np.log(10)

# Global x cutoff for BOTH plotting and AUC
X_CUTOFF = 400

# =========================
# DEFINE COLORS
# =========================
PRUNING_COLOR_MAP = {
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",  # 90%
    0.94:  "#BEBADA",  # new
    0.96:  "#80B1D3",  # new
    0.96:  "#FDB462",  # new
    0.98:  "#C5B0D5",  # new
    0.985:  "#6A5ACD",
    0.99:   "#FF6F61",
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]
# =========================
# HELPERS
# =========================
def fmt_p(p):
    s = f"{p:.3f}".rstrip("0").rstrip(".")
    if "." not in s:
        s += ".0"
    return s

def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    auc_table = []

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # -------------------------
        # LOAD ONLY SPECIFIED FILES
        # -------------------------
        for p in PRUNING_PERCENTAGES:
            p_str = fmt_p(p)
            folder = os.path.join(base_dir, f"p-percentage_{p_str}", f"batch_size_{bs}")
            file_path = os.path.join(
                folder,
                f"averaged_runs_conv_{prune_layer}_p_{p_str}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] File not found: {file_path}")
                continue

            df = pd.read_csv(file_path)

            df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
            df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

            avg_dfs[p] = df

        if not avg_dfs:
            print(f"No data found for {prune_layer}, batch size {bs}")
            continue

        # -------------------------
        # PLOT FUNCTION
        # -------------------------
        def plot_ce(avg_dfs, bs, ce_column, title):

            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | FMNIST | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(title)

            if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )

            # Plot in descending pruning order for legend display,
            # but colors are taken directly from the map.
            for p in sorted(avg_dfs.keys(), reverse=True):
                df = avg_dfs[p]
                color = PRUNING_COLOR_MAP.get(p, "#000000")
        
                if p == 1.0:
                    ce_value = df[ce_column].iloc[-1]
                    ce_value = min(ce_value, CE_MAX)
        
                    x = np.array([0, lowest_max_batch])
                    y = np.array([ce_value, ce_value])
                    auc = ce_value * lowest_max_batch
        
                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.5,
                        label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                    )

                else:
                    # Truncate strictly at x = 400 for BOTH plotting and AUC
                    df_trunc = df[df["Batch_Number"] <= X_CUTOFF]

                    x = df_trunc["Batch_Number"].to_numpy()
                    y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x_clean = x[mask]
                    y_clean = y[mask]

                    if len(x_clean) == 0:
                        print(f"[WARNING] No valid data up to Batch_Number={X_CUTOFF} for P%={p}")
                        continue

                    auc = np.trapezoid(y_clean, x_clean)

                    plt.plot(
                        x_clean, y_clean,
                        color=color,
                        linewidth=2.0,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(
                        x_clean, y_clean,
                        alpha=0.2,
                        color=color
                    )

                auc_table.append({
                    "Batch_Size": bs,
                    "Pruning_Percentage": p,
                    "CE_Type": ce_column,
                    "AUC": auc
                })

            plt.xlim(0, X_CUTOFF)
            plt.grid(True)
            plt.tight_layout()
            plt.legend(
                loc='center left',
                bbox_to_anchor=(1, 0.5),
                frameon=False
            )

            save_name = f"{prune_layer}_{title.replace(' ', '_')}_BS_{bs}.png"
            save_path = os.path.join(layer_graph_dir, save_name)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"[SAVED] {save_path}")

        plot_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE_Train")
        plot_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE_Test")

    if auc_table:
        auc_df = pd.DataFrame(auc_table)
        csv_path = os.path.join(layer_data_dir, f"AUC_table_{prune_layer}.csv")
        auc_df.to_csv(csv_path, index=False)
        print(f"[SAVED] AUC table: {csv_path}")

print("\nAll AUC graphs and tables generated successfully.")

[INFO] Graphs: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100
[INFO] AUC data: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_data_90-100

Processing prune layer: CONV
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\CONV\CONV_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\CONV\CONV_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\CONV\CONV_Average_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\CONV\CONV_Average_CE_Test_BS_1024.png
[SAVED] AUC table: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_data_90-100\CONV\AUC_table_CONV.csv

Processing prune layer: FHL
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\FHL\FHL_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_90-100\FHL\FHL_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\

In [44]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (Convolutional MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Micah\physlab\Convolutional-FMNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_SIZES = [64, 1024]

# ONLY these pruning percentages will be loaded/plotted
PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 1.0]

# =========================
# CREATE OUTPUT DIRECTORIES
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-90-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-90-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)
print(f"[INFO] Graphs: {AUC_GRAPH_DIR}")
print(f"[INFO] AUC data: {AUC_DATA_DIR}")

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# Maximum CE cap
CE_MAX = np.log(10)

# Global x cutoff for BOTH plotting and AUC
X_CUTOFF = 400

# =========================
# DEFINE COLORS
# =========================
PRUNING_COLOR_MAP = {
    0.0:  "#17becf",  # 0%
    0.1:  "#B9D9EB",  # 10%
    0.2:  "#bcbd22",  # 20%
    0.3:  "#7f7f7f",  # 30%
    0.4:  "#e377c2",  # 40%
    0.5:  "#8c564b",  # 50%
    0.6:  "#800080",  # 60%
    0.7:  "#d62728",  # 70%
    0.8:  "#2ca02c",  # 80%
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",  # 90%
    0.94:  "#BEBADA",  # new
    0.96:  "#80B1D3",  # new
    0.96:  "#FDB462",  # new
    0.98:  "#C5B0D5",  # new
    0.985:  "#6A5ACD",
    0.99:   "#FF6F61",
    1.0:    "#1f77b4"   # 100%
}

# Optional: keep this only if you want a list in the exact same order
COLOR_LIST = [PRUNING_COLOR_MAP[p] for p in PRUNING_PERCENTAGES]
# =========================
# HELPERS
# =========================
def fmt_p(p):
    s = f"{p:.3f}".rstrip("0").rstrip(".")
    if "." not in s:
        s += ".0"
    return s

def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    auc_table = []

    for bs in BATCH_SIZES:

        avg_dfs = {}

        # -------------------------
        # LOAD ONLY SPECIFIED FILES
        # -------------------------
        for p in PRUNING_PERCENTAGES:
            p_str = fmt_p(p)
            folder = os.path.join(base_dir, f"p-percentage_{p_str}", f"batch_size_{bs}")
            file_path = os.path.join(
                folder,
                f"averaged_runs_conv_{prune_layer}_p_{p_str}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] File not found: {file_path}")
                continue

            df = pd.read_csv(file_path)

            df["Avg_CE_Train"] = np.minimum(df["Avg_CE_Train"], CE_MAX)
            df["Avg_CE_Test"] = np.minimum(df["Avg_CE_Test"], CE_MAX)

            avg_dfs[p] = df

        if not avg_dfs:
            print(f"No data found for {prune_layer}, batch size {bs}")
            continue

        # -------------------------
        # PLOT FUNCTION
        # -------------------------
        def plot_ce(avg_dfs, bs, ce_column, title):

            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | FMNIST | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(title)

            if "CE" in ce_column:
                plt.text(
                    0.18,
                    CE_MAX,
                    r"$\ln(10)$",
                    transform=plt.gca().get_yaxis_transform(),
                    fontsize=12,
                    va="center",
                    ha="left",
                    bbox=dict(
                        facecolor='white',
                        edgecolor='none',
                        alpha=0.8,
                        pad=1
                    )
                )

            # Plot in descending pruning order for legend display,
            # but colors are taken directly from the map.
            for p in sorted(avg_dfs.keys(), reverse=True):
                df = avg_dfs[p]
                color = PRUNING_COLOR_MAP.get(p, "#000000")
        
                if p == 1.0:
                    ce_value = df[ce_column].iloc[-1]
                    ce_value = min(ce_value, CE_MAX)
        
                    x = np.array([0, lowest_max_batch])
                    y = np.array([ce_value, ce_value])
                    auc = ce_value * lowest_max_batch
        
                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.5,
                        label=f"P%={p*100:.0f} | AUC={auc:.2f}"
                    )

                else:
                    # Truncate strictly at x = 400 for BOTH plotting and AUC
                    df_trunc = df[df["Batch_Number"] <= X_CUTOFF]

                    x = df_trunc["Batch_Number"].to_numpy()
                    y = np.minimum(df_trunc[ce_column].to_numpy(), CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x_clean = x[mask]
                    y_clean = y[mask]

                    if len(x_clean) == 0:
                        print(f"[WARNING] No valid data up to Batch_Number={X_CUTOFF} for P%={p}")
                        continue

                    auc = np.trapezoid(y_clean, x_clean)

                    plt.plot(
                        x_clean, y_clean,
                        color=color,
                        linewidth=2.0,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(
                        x_clean, y_clean,
                        alpha=0.2,
                        color=color
                    )

                auc_table.append({
                    "Batch_Size": bs,
                    "Pruning_Percentage": p,
                    "CE_Type": ce_column,
                    "AUC": auc
                })

            plt.xlim(0, X_CUTOFF)
            plt.grid(True)
            plt.tight_layout()
            plt.legend(
                loc='center left',
                bbox_to_anchor=(1, 0.5),
                frameon=False
            )

            save_name = f"{prune_layer}_{title.replace(' ', '_')}_BS_{bs}.png"
            save_path = os.path.join(layer_graph_dir, save_name)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"[SAVED] {save_path}")

        plot_ce(avg_dfs, bs, "Avg_CE_Train", "Average CE_Train")
        plot_ce(avg_dfs, bs, "Avg_CE_Test", "Average CE_Test")

    if auc_table:
        auc_df = pd.DataFrame(auc_table)
        csv_path = os.path.join(layer_data_dir, f"AUC_table_{prune_layer}.csv")
        auc_df.to_csv(csv_path, index=False)
        print(f"[SAVED] AUC table: {csv_path}")

print("\nAll AUC graphs and tables generated successfully.")

[INFO] Graphs: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100
[INFO] AUC data: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_data_0-90-100

Processing prune layer: CONV
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\CONV\CONV_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\CONV\CONV_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\CONV\CONV_Average_CE_Train_BS_1024.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\CONV\CONV_Average_CE_Test_BS_1024.png
[SAVED] AUC table: C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_data_0-90-100\CONV\AUC_table_CONV.csv

Processing prune layer: FHL
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\FHL\FHL_Average_CE_Train_BS_64.png
[SAVED] C:\Users\Micah\physlab\Convolutional-FMNIST\AUC_graph_0-90-100\FHL\FHL_Average_CE_Test_BS_64.png
[SAVED] C:\Users\Micah\physlab\Con